In [35]:
import numpy as np

In [36]:

def expected_new_places(state, action, layout, circle):

    def rolls_security_dice():
        possible_trap_triggered = [False]
        possible_dice_rolls = [0, 1]
        p = 1/2
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    def rolls_normal_dice():
        possible_trap_triggered = [True, False]
        possible_dice_rolls = [0, 1, 2]
        p = 1/6
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    


    def rolls_risky_dice():
        possible_trap_triggered = [True]
        possible_dice_rolls = [0, 1, 2, 3]
        p = 1/4
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    rolls_dice_functions = {0:rolls_security_dice, 1:rolls_normal_dice, 2:rolls_risky_dice} 



    (current_position, current_skip_next_turn) = state

    
    if current_skip_next_turn:
        new_state = (current_position, False)
        return [(1, new_state)]
    

    # roll dices
    roll_function = rolls_dice_functions[action]
    list_rolls =  roll_function()

    list_new_positions_before_traps = []

    for (p, trap_triggered, dice_roll) in list_rolls:
        # find new position
        if dice_roll==0:
            new_position_before_trap = current_position
            list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))
        else: # dice roll is not 0
            if (current_position == 2):
                    new_position_before_trap = current_position + dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))
                    new_position_before_trap = 9+dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))


            elif  current_position in range(10): # but not 2
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 10:
                    if circle:
                        new_position_before_trap -= 10
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

            elif current_position in range(10, 14):
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 14:
                    if circle:
                        new_position_before_trap -=14
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

    list_new_places_after_traps = []

    for (p, new_position_before_trap, trap_triggered) in list_new_positions_before_traps:
        # Deal with the traps
        trap = layout[new_position_before_trap]

        if  (not trap_triggered) or (trap == 0):
            new_skip_next_turn = False
            new_position_after_trap = new_position_before_trap
            list_new_places_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))
        else:
            trap_list = []

            if trap == 4:
                trap_list.append(1)
                trap_list.append(2)
                trap_list.append(3)
                p/=3
            else:
                trap_list.append(trap)
            
            for trap in trap_list:
                if   trap == 1:
                    new_position_after_trap = 0
                    new_skip_next_turn = False

                elif trap == 2:
                    new_position_after_trap = new_position_before_trap
                    if new_position_after_trap in range(10, 13):
                        new_position_after_trap -= 7 # -7 -3 = -10
                    new_position_after_trap = max(0, new_position_after_trap - 3)
                    new_skip_next_turn = False


                elif trap == 3:
                    new_position_after_trap = new_position_before_trap
                    new_skip_next_turn=True

                list_new_places_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))

    return list_new_places_after_traps




def expected_probability_win(state, action, P, layout, circle):
    # state = (position1:(), positions2:(), whoseturn)
    # if whoseturn==1:-> min
    # if whoseturn==0:-> max

    (my_place, other_place, whoseturn) = state

    if other_place[0] == 14:
        if whoseturn: return 1
        else: return 0
    
    list_expected_my_new_places = expected_new_places(my_place, action, layout, circle)

    new_probability  = 0
    for (p, new_place) in list_expected_my_new_places:
        new_probability += p*P[(other_place, new_place, not whoseturn)]

    return new_probability

    
    
def min_max(layout, circle, theta):
    # if whoseturn==1:-> min
    # if whoseturn==0:-> max

    possible_places = []
    for i in range(14):
        if layout[i]>=3:
            possible_places.append((i, False))
            possible_places.append((i, True))
        else:
            possible_places.append((i, False))

    possible_states = []
    for place1 in possible_places:
        for place2 in possible_places:
            possible_states.append((place1, place2, False))
            possible_states.append((place1, place2, True))

        possible_states.append((place1, (14, False), False))
        possible_states.append((place1, (14, False),  True))


    P = {}
    best_policy = {}

    for state in possible_states:
        P[state] = 0
        best_policy[state] = 0
    
    delta = 2*theta
    while delta>=theta:
        print(delta)
        delta = 0
        for state in possible_states:
            v = P[state]
            (_, _, whoseturn) = state

            if whoseturn:
                minv = np.inf
                for action in range(3):
                    Pn = expected_probability_win(state, action, P, layout, circle)
                    # print(nv)
                    if Pn <= minv:
                        minv = Pn
                        best_policy[state] = action
                P[state] = minv
            else : 
                maxv = -np.inf
                for action in range(3):
                    Pn = expected_probability_win(state, action, P, layout, circle)
                    if Pn >= maxv:
                        maxv = Pn
                        best_policy[state] = action
                P[state] = maxv
            delta = max(abs(v-P[state]), delta)
    print(delta)
    return P, best_policy
        


In [44]:
# circle: a boolean variable (type bool), indicating if the player must land exactly on
# the final, goal, square 15 to win (circle = True) or still wins by overstepping the final
# square (circle = False).
circle = True

# layout: a vector of type numpy.ndarray that represents the layout of the game, containing 15 values
#         representing the 15 squares of the Snakes and Ladders game:
# layout[i] = 0 if it is an ordinary square
#           = 1 if it is a “restart” trap (go back to square 1)
#           = 2 if it is a “penalty” trap (go back 3 steps)
#           = 3 if it is a “prison” trap (skip next turn)
#           = 4 if it is a “mystery” trap (random effect among the three previous)
# Note that the first and final squares cannot be trapped.

layout = np.ones(15)*4
layout[0] = 0
layout[14] = 0

min_max(layout, circle, 0.0001)



dict_keys([((0, False), (0, False), False), ((0, False), (0, False), True), ((0, False), (1, False), False), ((0, False), (1, False), True), ((0, False), (1, True), False), ((0, False), (1, True), True), ((0, False), (2, False), False), ((0, False), (2, False), True), ((0, False), (2, True), False), ((0, False), (2, True), True), ((0, False), (3, False), False), ((0, False), (3, False), True), ((0, False), (3, True), False), ((0, False), (3, True), True), ((0, False), (4, False), False), ((0, False), (4, False), True), ((0, False), (4, True), False), ((0, False), (4, True), True), ((0, False), (5, False), False), ((0, False), (5, False), True), ((0, False), (5, True), False), ((0, False), (5, True), True), ((0, False), (6, False), False), ((0, False), (6, False), True), ((0, False), (6, True), False), ((0, False), (6, True), True), ((0, False), (7, False), False), ((0, False), (7, False), True), ((0, False), (7, True), False), ((0, False), (7, True), True), ((0, False), (8, False), Fal

({((0, False), (0, False), False): 0.5172423003509374,
  ((0, False), (0, False), True): 0.48216840955106566,
  ((0, False), (1, False), False): 0.4647752282860016,
  ((0, False), (1, False), True): 0.5347783550352028,
  ((0, False), (1, True), False): 0.5003312888951319,
  ((0, False), (1, True), True): 0.49916851628361425,
  ((0, False), (2, False), False): 0.39437794229418055,
  ((0, False), (2, False), True): 0.6053176814752601,
  ((0, False), (2, True), False): 0.42923972033674507,
  ((0, False), (2, True), True): 0.5704195071978905,
  ((0, False), (3, False), False): 0.5480015876778074,
  ((0, False), (3, False), True): 0.45153734011423813,
  ((0, False), (3, True), False): 0.5829944534193395,
  ((0, False), (3, True), True): 0.41650748713388175,
  ((0, False), (4, False), False): 0.5315443262492476,
  ((0, False), (4, False), True): 0.4681508338019531,
  ((0, False), (4, True), False): 0.5681004881891647,
  ((0, False), (4, True), True): 0.43156740824965045,
  ((0, False), (5, F